# Data Engineering for AI : de la donnée brute à un dataset de confiance

Atelier TAISS 2026 — jeudi 27 août, 13h30 à 15h30, salle F1. Animateur : Babacar Ndao, Afriklang.

Un seul jeu de données traverse les deux heures : le manifeste d'un corpus vocal, 200 lignes qui décrivent des enregistrements en français et en éwé. Vous allez l'inspecter, le nettoyer, l'annoter, mesurer la fiabilité de vos annotations, et repartir avec un dataset documenté :

```
raw_manifest.csv → clean_manifest.csv → annotated_manifest.csv → dataset final
```

## Section 1 — Le pipeline (13h30, 10 min)

> **Consigne (10 min)** — Suivre la présentation. Exécuter les deux cellules ci-dessous pendant la distribution du lien : la première met les données en place, la seconde vérifie que tout est lisible.

```
Collecte → Inspection → Nettoyage → Annotation → Contrôle qualité → Dataset
```

Toutes ces étapes comptent. Une seule échoue en silence, et c'est celle que cet atelier prend au sérieux.

Pourquoi travailler sur un manifeste plutôt que sur les fichiers audio : dans un pipeline réel, les contrôles automatiques tournent d'abord sur le tableau qui décrit les enregistrements, et seuls les cas signalés partent en écoute humaine. Écouter dix mille fichiers n'est pas une option. Filtrer dix mille lignes puis écouter les deux cents suspectes en est une.

In [ ]:
# Mise en place des données. Trois cas, dans cet ordre :
# 1. les fichiers sont déjà là (poste local ou session déjà préparée)
# 2. téléchargement depuis l'URL publique communiquée par l'animateur
# 3. repli : téléversement manuel du zip dans Colab
import os
import zipfile
from pathlib import Path
from urllib.request import urlretrieve

DATA_URL = ""  # communiquée au tableau si le téléchargement est nécessaire

if Path("data/raw_manifest.csv").exists():
    print("données en place, rien à télécharger")
elif Path("taiss2026_workshop/data/raw_manifest.csv").exists():
    os.chdir("taiss2026_workshop")
    print("données trouvées, dossier de travail :", Path.cwd())
elif DATA_URL:
    try:
        urlretrieve(DATA_URL, "taiss2026_workshop.zip")
        with zipfile.ZipFile("taiss2026_workshop.zip") as z:
            z.extractall(".")
        os.chdir("taiss2026_workshop")
        print("données téléchargées, dossier de travail :", Path.cwd())
    except Exception as e:
        print("téléchargement impossible :", e)
        print("repli : menu Fichiers de Colab, téléverser le zip fourni,")
        print("puis relancer cette cellule")
else:
    print("pas de données locales et pas d'URL : téléverser le zip fourni")
    print("dans le menu Fichiers de Colab, puis relancer cette cellule")

# le dossier que vous emporterez se construit au fil de la séance
DATASET_DIR = Path("taiss2026_sentiment_ee-fr_v1.0")
for sub in ["data", "splits", "scripts"]:
    (DATASET_DIR / sub).mkdir(parents=True, exist_ok=True)

In [ ]:
import sys
import pandas as pd

sys.path.insert(0, "scripts")

raw = pd.read_csv("data/raw_manifest.csv", encoding="utf-8")
print(raw.shape[0], "lignes,", raw.shape[1], "colonnes")
raw.head()

## Section 2 — Inspecter un manifeste dégradé (13h40, 13 min)

> **Consigne (8 min, en binômes)** — Trouvez tout ce qui ne va pas avec ce manifeste. Notez chaque famille de problème que vous repérez, avec un exemple de ligne. La restitution se fait au tableau, gardez vos trouvailles pour vous d'ici là.

Vous venez d'entendre trois problèmes au haut-parleur. Il y en a davantage dans ce tableau, et vous ne pourrez écouter aucun fichier : c'est la situation normale d'un pipeline de données vocales.

In [ ]:
# points de départ, à compléter par vos propres idées
raw.describe(include="all").T

In [ ]:
raw[["language", "sample_rate", "channels", "region", "source"]].apply(
    lambda col: col.value_counts(dropna=False).to_dict())

In [ ]:
raw.isna().sum()

In [ ]:
# votre exploration


Ce que ce tableau ne montrera jamais, quel que soit le soin de l'inspection : si les labels d'annotation qu'on posera dessus seront fiables. Les défauts structurels se voient en dix minutes. Le défaut d'annotation ne se voit nulle part, et c'est la raison d'être de la séquence 4.

## Section 3 — Écrire les règles de qualité (13h53, 12 min)

> **Consigne (12 min)** — Quatre fonctions de contrôle à écrire, une cellule chacune, puis la fonction d'agrégation. Chaque cellule contient le contrat, l'exemple et ses tests : exécutez la cellule pour savoir où vous en êtes. Ceux qui finissent en avance affinent leurs seuils et regardent ce que ça change au rapport final.

Deux idées gouvernent cette section.

Un seuil est une décision, pas une vérité. Fixer la durée maximale à 20 secondes plutôt qu'à 30 change le dataset, donc le modèle qu'on entraînera dessus. Ces décisions sont écrites en tête de `scripts/quality_check.py`, visibles et discutables, jamais enfouies dans le code.

Un filtre signale, il ne prouve pas. Le contrôle de débit de parole ne démontre pas qu'une transcription est fausse : il désigne l'enregistrement qu'un humain doit écouter, comme le troisième fichier joué en ouverture.

In [ ]:
from quality_check import (
    check_duration, check_sample_rate, check_transcription_present,
    run_tests, TEST_CASES, _text, _words,
    FRENCH_STOPWORDS, REQUIRED_METADATA_FIELDS, REFERENCE_DATE,
    EXPECTED_CHANNELS, EXPECTED_SAMPLE_RATE, MAX_WORDS_PER_SECOND,
    MIN_DURATION_S, MAX_DURATION_S,
)

# trois contrôles sont déjà écrits, à lire comme des modèles
import inspect
print(inspect.getsource(check_duration))

### Exercice — `check_channels`

**Attendu** : True si la ligne est en mono (`channels == 1`), False sinon, y compris quand la valeur est absente ou illisible.

**Le test vérifie** : mono accepté, stéréo refusé, valeur non numérique et valeur absente refusées.

**Si vous êtes bloqué** : `check_sample_rate` fait exactement le même travail sur une autre colonne, lisez-la.

In [ ]:
def check_channels(row, expected=EXPECTED_CHANNELS):
    """Contrôle que le nombre de canaux est exactement celui attendu.

    Entrée : une ligne du manifeste (row["channels"]).
    Sortie : True si channels == expected, False sinon.
    Cas limites : valeur absente ou non numérique → False.

    Exemple :
        >>> check_channels({"channels": 1})
        True
        >>> check_channels({"channels": 2})
        False
    """
    # TODO : sur le modèle de check_sample_rate.
    raise NotImplementedError


run_tests(check_channels)

### Exercice — `check_words_per_second`

**Attendu** : False quand le nombre de mots divisé par la durée dépasse `max_wps`, True sinon. Une transcription vide ou une durée invalide ne sont pas le problème de ce contrôle : True dans ces cas, d'autres contrôles s'en chargent.

**Le test vérifie** : le cas 40 mots en 0,6 s, les cas vides, la durée nulle, et un débit juste au-dessus du seuil.

**Si vous êtes bloqué** : comptez les mots avec `_words(...)`, convertissez la durée comme le fait `check_duration`, comparez le rapport au seuil. Attention au sens : True veut dire conforme.

In [ ]:
def check_words_per_second(row, max_wps=MAX_WORDS_PER_SECOND):
    """Contrôle que le débit de parole annoncé est physiquement plausible.

    Entrée : une ligne du manifeste (row["transcription"], row["duration_s"]).
    Sortie : False si nombre_de_mots / durée > max_wps, True sinon.
    Cas limites : transcription vide → True (0 mot, débit nul, c'est le rôle
    de check_transcription_present de signaler le vide). Durée nulle, négative
    ou illisible → True, c'est le rôle de check_duration de la signaler ;
    ce contrôle ne juge que le rapport entre les deux.

    Ce contrôle SIGNALE une transcription à vérifier, il ne prouve pas
    qu'elle est fausse. Un débit impossible veut dire : quelqu'un doit
    écouter ce fichier.

    Exemple :
        >>> check_words_per_second({"transcription": "quarante mots " * 20,
        ...                         "duration_s": 0.6})
        False
    """
    # TODO : compter les mots avec _words(...), convertir la durée comme dans
    # check_duration, comparer le rapport au seuil.
    raise NotImplementedError


run_tests(check_words_per_second)

### Exercice — `check_language_consistency`

**Attendu** : False pour une ligne marquée `ee` dont le texte contient au moins deux mots outils français distincts, True pour tout le reste. Les lignes `fr` sont toujours acceptées : sans lexique éwé, l'inverse n'est pas contrôlable, et le code l'assume plutôt que de le cacher.

**Le test vérifie** : une ligne `ee` en français est refusée, une vraie ligne éwé passe, une ligne vide passe.

**Si vous êtes bloqué** : `_words(...)` découpe, `FRENCH_STOPWORDS` est un ensemble, l'intersection de deux ensembles se compte avec `len`.

In [ ]:
def check_language_consistency(row):
    """Contrôle que le texte d'une ligne marquée « ee » ressemble bien à de l'éwé.

    Heuristique, pas détection : on compte les mots outils français distincts
    dans la transcription (lexique FRENCH_STOPWORDS). Deux ou plus dans une
    ligne marquée « ee » → la ligne est suspecte → False. Une ligne marquée
    « fr » est toujours acceptée : sans lexique éwé, l'inverse n'est pas
    contrôlable, et on le dit ici plutôt que de le laisser croire.

    Entrée : une ligne du manifeste (row["language"], row["transcription"]).
    Sortie : True si cohérent (ou incontrôlable), False si une ligne « ee »
    contient au moins deux mots outils français distincts.
    Cas limites : transcription vide → True.

    Exemple :
        >>> check_language_consistency({"language": "ee",
        ...     "transcription": "Les enfants sont dans la cour pour jouer."})
        False
    """
    # TODO : ne traiter que le cas language == "ee". Tokeniser avec _words,
    # compter les mots distincts présents dans FRENCH_STOPWORDS.
    raise NotImplementedError


run_tests(check_language_consistency)

### Exercice — `check_metadata_complete`

**Attendu** : True quand `speaker_id`, `region`, `recorded_at` et `source` sont tous non vides ET que la date d'enregistrement ne dépasse pas le jour de l'atelier. Un enregistrement daté du futur est impossible.

**Le test vérifie** : la ligne complète passe, chaque champ vide ou NaN fait échouer, une date de 2027 fait échouer.

**Si vous êtes bloqué** : `_text(...)` rend toute valeur comparable à la chaîne vide, et deux dates au format AAAA-MM-JJ se comparent directement entre chaînes.

In [ ]:
def check_metadata_complete(row):
    """Contrôle que les métadonnées minimales sont présentes et plausibles.

    Entrée : une ligne du manifeste.
    Sortie : True si chaque champ de REQUIRED_METADATA_FIELDS est non vide
    (au sens de _text) ET si recorded_at ne dépasse pas REFERENCE_DATE,
    False sinon.
    Cas limites : un enregistrement daté d'après le jour de l'atelier est
    impossible, donc non conforme. La comparaison de dates au format
    AAAA-MM-JJ peut se faire directement entre chaînes.

    Exemple :
        >>> check_metadata_complete({"speaker_id": "spk_012", "region": "Kara",
        ...     "recorded_at": "2026-01-15", "source": "studio_lome"})
        True
    """
    # TODO : _text(...) pour tester la présence, comparaison de chaînes pour
    # la date.
    raise NotImplementedError


run_tests(check_metadata_complete)

### Exercice — `quality_gate`

**Attendu** : la fonction appelle chaque contrôle de la liste et retourne `(accepted, reasons)` : `accepted` vaut True seulement si tout passe, `reasons` liste les noms des contrôles en échec, dans l'ordre de la liste.

**Le test vérifie** : ligne conforme → `(True, [])`, une puis deux pannes → les bons noms dans le bon ordre.

**Si vous êtes bloqué** : `fn.__name__` donne le nom d'une fonction, et une liste en compréhension avec un `if` suffit.

In [ ]:
ALL_CHECKS = [
    check_duration, check_sample_rate, check_transcription_present,
    check_channels, check_words_per_second, check_language_consistency,
    check_metadata_complete,
]


def quality_gate(row, checks=None):
    """Agrège les contrôles sur une ligne.

    Entrée : une ligne du manifeste, et optionnellement une liste de fonctions
    de contrôle (par défaut ALL_CHECKS).
    Sortie : (accepted, reasons) où accepted vaut True si tous les contrôles
    passent, et reasons liste le nom des contrôles qui ont échoué
    (fonction.__name__). Ligne conforme → (True, []).

    Exemple :
        >>> quality_gate({"duration_s": 3.0, "sample_rate": 8000, ...})
        (False, ['check_sample_rate'])
    """
    if checks is None:
        checks = ALL_CHECKS
    reasons = []
    # TODO : appeler chaque contrôle, collecter le nom de ceux qui échouent,
    # en déduire accepted.
    raise NotImplementedError


run_tests(quality_gate)

La cellule suivante applique la barrière de qualité aux 200 lignes. Elle n'attend pas que tout soit résolu : un contrôle dont les tests ne passent pas encore est simplement laissé de côté, la séance continue, et le rapport dit lesquels ont tourné. Revenez remplir les trous quand vous voulez, puis relancez-la.

In [ ]:
import contextlib
import io
import re


def _tests_passent(fn):
    with contextlib.redirect_stdout(io.StringIO()):
        try:
            return run_tests(fn)
        except Exception:
            return False


checks_actifs = [check_duration, check_sample_rate, check_transcription_present]
checks_ignores = []
for fn in [check_channels, check_words_per_second,
           check_language_consistency, check_metadata_complete]:
    (checks_actifs if _tests_passent(fn) else checks_ignores).append(fn)

erreurs_execution = {}


def gate_robuste(row):
    reasons = []
    for fn in checks_actifs:
        try:
            if not fn(row):
                reasons.append(fn.__name__)
        except Exception:
            # un contrôle qui plante sur une vraie ligne ne bloque pas la
            # séance : on le compte et on continue
            erreurs_execution[fn.__name__] = erreurs_execution.get(fn.__name__, 0) + 1
    return len(reasons) == 0, reasons


verdicts = raw.apply(gate_robuste, axis=1)
raw_checked = raw.copy()
raw_checked["accepted"] = [v[0] for v in verdicts]
raw_checked["reasons"] = [";".join(v[1]) for v in verdicts]

# deux opérations niveau tableau complètent les contrôles ligne à ligne :
# les doublons et l'encodage cassé ne se voient pas depuis une ligne seule


def _norme(s):
    return re.sub(r"\s+", " ", str(s).strip().lower())


doublon = raw.astype(str).apply(lambda c: c.map(_norme)).duplicated(keep="first")
moji = raw["transcription"].fillna("").str.contains("Ã|â€", regex=True)

garde = raw_checked["accepted"] & ~doublon & ~moji
clean = raw[garde].copy()
clean.to_csv(DATASET_DIR / "data" / "clean_manifest.csv",
             index=False, encoding="utf-8")

motifs = {}
for _, v in raw_checked[~raw_checked["accepted"]].iterrows():
    for m in v["reasons"].split(";"):
        motifs[m] = motifs.get(m, 0) + 1

lignes_rapport = [
    "# Rapport de qualité — raw_manifest.csv", "",
    f"Lignes en entrée : {len(raw)}",
    f"Lignes conservées : {len(clean)}",
    f"Lignes rejetées par les contrôles : {int((~raw_checked.accepted).sum())}",
    f"Doublons retirés (exacts et approximatifs) : {int(doublon.sum())}",
    f"Encodage cassé (mojibake) : {int(moji.sum())}", "",
    "Motifs de rejet :",
] + [f"- {k} : {v}" for k, v in sorted(motifs.items())] + [
    "",
    f"Contrôles actifs : {[f.__name__ for f in checks_actifs]}",
    f"Contrôles ignorés (tests non passés) : {[f.__name__ for f in checks_ignores]}",
    f"Seuils : durée [{MIN_DURATION_S}, {MAX_DURATION_S}] s, "
    f"{EXPECTED_SAMPLE_RATE} Hz, mono, {MAX_WORDS_PER_SECOND} mots/s max",
]
if erreurs_execution:
    lignes_rapport.append(f"Contrôles ayant levé des erreurs en cours de "
                          f"route : {erreurs_execution}")
(DATASET_DIR / "quality_report.md").write_text(
    "\n".join(lignes_rapport), encoding="utf-8")

print(f"{len(raw)} lignes en entrée, {len(clean)} conservées")
print(f"contrôles actifs : {[f.__name__ for f in checks_actifs]}")
if checks_ignores:
    print(f"contrôles ignorés pour l'instant : "
          f"{[f.__name__ for f in checks_ignores]}")
print("rapport écrit :", DATASET_DIR / "quality_report.md")

## Section 4 — Mesurer la fiabilité des labels (14h05, 45 min)

> **Consigne (5 min pour cette page)** — Lire le guide d'annotation v1 distribué (`guides/annotation_guide_v1_fr.md` ou sa version éwé selon votre groupe). Pas de questions sur les cas particuliers : le guide est réputé suffisant, c'est lui votre seule référence pour le round 1.

La tâche : trois classes de sentiment, `positif`, `negatif`, `neutre`, sur la colonne `transcription`. Chaque binôme annote la même série de phrases, chacun de son côté.

Une objection arrive toujours ici : le sentiment est subjectif, le désaccord serait donc normal. C'est exactement pourquoi il faut un guide. Un guide ne supprime pas la subjectivité, il la rend reproductible : deux annotateurs qui appliquent la même règle explicite convergent, même sur une tâche subjective. C'est ce que le Kappa mesure.

### Round 1 — annotation en aveugle (13 min)

> **Consigne (13 min)** — 40 phrases, chacun annote seul, sans se concerter avec son binôme. Si vous vous alignez, la mesure ne veut plus rien dire. Saisie dans l'onglet de votre binôme du Google Sheet projeté, colonnes `annotateur_A` et `annotateur_B`, ou sur papier si la connexion tombe.

In [ ]:
LANGUE = "fr"  # passer à "ee" pour le groupe éwé

round1 = pd.read_csv(f"data/transcriptions_{LANGUE}_round1.csv",
                     encoding="utf-8")
if round1["transcription"].str.contains(r"\[À REMPLIR", na=False).any():
    print("le corpus éwé n'est pas encore chargé dans ce kit ;")
    print("le groupe éwé travaille sur la version distribuée en salle")
else:
    with pd.option_context("display.max_colwidth", None):
        display(round1)

### Mesurer l'accord (9 min)

> **Consigne (9 min)** — Récupérer vos labels ci-dessous, par le Sheet ou en les collant à la main, puis exécuter les cellules de mesure. Reportez votre Kappa au tableau dans la colonne de votre groupe.

L'accord brut ment. Deux annotateurs qui répondent `neutre` partout obtiennent un accord brut spectaculaire et n'ont rien jugé du tout : le Kappa corrige l'accord de ce que le hasard produirait compte tenu des habitudes de chacun. C'est l'écart entre les deux chiffres qui va vous surprendre.

In [ ]:
# chemin principal : le Google Sheet de votre binôme, publié au format CSV.
# Fichier → Partager → Publier sur le web → votre onglet → CSV, puis coller
# l'URL ici. Colonnes attendues : id, annotateur_A, annotateur_B.
SHEET_CSV_URL_R1 = ""

# chemin de repli : coller les deux listes à la main, dans l'ordre du
# tableau affiché plus haut
labels_A = []  # coller ici vos 40 labels, ex. "positif", "neutre", ...
labels_B = []

In [ ]:
from agreement import (normalize_labels, raw_agreement, kappa,
                       confusion, disagreements, compare_guides, summary)


def recuperer_labels(sheet_url, manuel_a, manuel_b, ids_attendus):
    """Choisit le chemin disponible et rend deux listes de labels propres.

    Priorité au Sheet s'il est renseigné et lisible, sinon aux listes
    manuelles si elles sont remplies. Retourne (None, None) plutôt que de
    lever une erreur : la suite du notebook saute proprement ce qui manque.
    """
    if sheet_url:
        try:
            feuille = pd.read_csv(sheet_url)
            manquantes = {"id", "annotateur_A", "annotateur_B"} - set(feuille.columns)
            if manquantes:
                print("colonnes manquantes dans le Sheet :", manquantes)
            else:
                feuille = feuille.set_index("id").reindex(ids_attendus)
                absents = feuille["annotateur_A"].isna() | feuille["annotateur_B"].isna()
                if absents.any():
                    print(f"{int(absents.sum())} lignes sans label dans le Sheet, "
                          f"ids : {list(feuille.index[absents])[:5]}...")
                else:
                    a, b = list(feuille["annotateur_A"]), list(feuille["annotateur_B"])
                    print("labels récupérés depuis le Sheet")
                    return _nettoyer(a, b)
        except Exception as e:
            print("Sheet illisible :", e)
    if manuel_a and manuel_b:
        if len(manuel_a) != len(ids_attendus) or len(manuel_b) != len(ids_attendus):
            print(f"il faut {len(ids_attendus)} labels par annotateur, reçu "
                  f"{len(manuel_a)} et {len(manuel_b)}")
            return None, None
        print("labels récupérés depuis les listes manuelles")
        return _nettoyer(manuel_a, manuel_b)
    print("pas encore de labels : renseigner SHEET_CSV_URL ou les listes,")
    print("puis relancer cette cellule ; la suite du notebook reste utilisable")
    return None, None


def _nettoyer(a, b):
    a, anom_a = normalize_labels(a)
    b, anom_b = normalize_labels(b)
    for nom, anomalies in (("A", anom_a), ("B", anom_b)):
        if anomalies:
            print(f"labels illisibles chez {nom} : {anomalies} — corrigez et relancez")
            return None, None
    return a, b


labels_A_r1, labels_B_r1 = recuperer_labels(
    SHEET_CSV_URL_R1, labels_A, labels_B, list(round1["id"]))

In [ ]:
if labels_A_r1 is not None:
    mesures_r1 = summary(labels_A_r1, labels_B_r1)
    print(mesures_r1)
    kappa_v1 = mesures_r1["kappa"]
    display(confusion(labels_A_r1, labels_B_r1))
else:
    kappa_v1 = None

### Diagnostiquer et réécrire le guide (12 min)

> **Consigne (12 min, collectif)** — Lire les désaccords ci-dessous à voix haute, phrase par phrase. Pour chacun : est-ce une faute d'inattention, ou bien le guide ne disait rien ? Chaque silence du guide identifié au tableau devient une règle explicite. L'ensemble de ces règles est votre guide v2, à recopier dans la cellule prévue plus bas.

In [ ]:
if labels_A_r1 is not None:
    table_desaccords = disagreements(round1, labels_A_r1, labels_B_r1)
    with pd.option_context("display.max_colwidth", None):
        display(table_desaccords)
else:
    print("pas de labels round 1, rien à diagnostiquer pour l'instant")

In [ ]:
GUIDE_V2 = """
# Guide d'annotation — Sentiment (version 2, écrite par la salle)

Reprend le guide v1, plus les règles décidées collectivement :

## Règle 1 —

## Règle 2 —

## Règle 3 —

## Règle 4 —

## Règle 5 —

## Règle 6 —
"""

(DATASET_DIR / "annotation_guide_v2.md").write_text(GUIDE_V2, encoding="utf-8")
print("guide v2 sauvegardé dans", DATASET_DIR / "annotation_guide_v2.md")

### Round 2 — mêmes annotateurs, nouveau guide (6 min)

> **Consigne (6 min)** — 20 nouvelles phrases, mêmes binômes, même interdiction de se concerter. Une seule chose a changé : vous annotez avec le guide v2. Saisie dans les colonnes round 2 de votre onglet.

In [ ]:
round2 = pd.read_csv(f"data/transcriptions_{LANGUE}_round2.csv",
                     encoding="utf-8")
if not round2["transcription"].str.contains(r"\[À REMPLIR", na=False).any():
    with pd.option_context("display.max_colwidth", None):
        display(round2)

SHEET_CSV_URL_R2 = ""
labels_A_2 = []  # les 20 labels du round 2
labels_B_2 = []

labels_A_r2, labels_B_r2 = recuperer_labels(
    SHEET_CSV_URL_R2, labels_A_2, labels_B_2, list(round2["id"]))

In [ ]:
if labels_A_r2 is not None:
    mesures_r2 = summary(labels_A_r2, labels_B_r2)
    print(mesures_r2)
    kappa_v2 = mesures_r2["kappa"]
else:
    kappa_v2 = None

if kappa_v1 is not None and kappa_v2 is not None:
    compare_guides(kappa_v1, kappa_v2)

Rien n'a changé dans les annotateurs ni dans le type de phrases, la proportion de cas difficiles est la même dans les deux rounds. Seul le guide a changé. Si votre Kappa monte, vous venez de voir la démonstration entière de l'atelier tenir dans un chiffre. S'il ne monte pas dans votre binôme : vingt items, c'est un petit échantillon, et cette instabilité est précisément la raison pour laquelle un jeu d'évaluation de production en compte des milliers.

## Section 5 — Pré-annotation IA et validation humaine (14h50, 15 min)

> **Consigne (15 min)** — Exécuter les cellules, puis discussion : où le modèle se trompe-t-il, et sur quelles classes ? La démonstration en direct se fait depuis le poste de l'animateur, pas sur vos machines.

Un modèle a annoté les mêmes phrases que vous, en zéro-shot : trois classes demandées, aucune règle de cas limite fournie, exactement votre situation du round 1. Ses prédictions sont livrées pré-calculées dans `data/model_predictions.csv`.

In [ ]:
predictions = pd.read_csv("data/model_predictions.csv", encoding="utf-8")
pred_langue = predictions[predictions["language"] == LANGUE]
if pred_langue.empty:
    print(f"pas encore de prédictions pour la langue {LANGUE!r} ;")
    print("la passe éwé tourne dès que le corpus est chargé")
else:
    print(len(pred_langue), "prédictions du modèle pour", LANGUE)
    display(pred_langue["predicted_label"].value_counts())

In [ ]:
# votre référence humaine : les phrases où votre binôme est d'accord.
# En production on utiliserait un gold arbitré par un troisième annotateur ;
# en deux heures, le consensus du binôme en tient lieu, et cette
# simplification est exactement le genre de choix qu'une dataset card doit
# documenter.
consensus = {}
for tableau, la, lb in ((round1, labels_A_r1, labels_B_r1),
                        (round2, labels_A_r2, labels_B_r2)):
    if la is None:
        continue
    for rid, a, b in zip(tableau["id"], la, lb):
        if a == b:
            consensus[rid] = a

if not consensus or pred_langue.empty:
    accord_modele = None
    print("comparaison sautée : il faut des labels de binôme et des "
          "prédictions dans votre langue")
else:
    commun = pred_langue[pred_langue["id"].isin(consensus)]
    verdicts = [consensus[r.id] == r.predicted_label
                for r in commun.itertuples()]
    accord_modele = sum(verdicts) / len(verdicts)
    print(f"accord du modèle avec votre consensus : {accord_modele:.0%} "
          f"sur {len(verdicts)} phrases")
    rates = commun[[not v for v in verdicts]].merge(
        pd.concat([round1, round2])[["id", "transcription"]], on="id")
    rates["votre_label"] = rates["id"].map(consensus)
    with pd.option_context("display.max_colwidth", None):
        display(rates[["id", "transcription", "votre_label", "predicted_label"]])

Regardez sur quelles phrases le modèle décroche : ce sont massivement les cas limites que votre guide v2 vient de trancher. Le modèle applique ses propres conventions implicites, pas les vôtres, et aucun F1 ne le dira si le jeu d'évaluation a été construit avec les mêmes conventions implicites.

D'où l'architecture qui structure le travail d'Afriklang :

```
Pré-annotation IA → Validation humaine → Contrôle qualité → Dataset
```

Sur une tâche d'analyse de sentiment en wolof, avec un protocole documenté, les annotateurs natifs d'Afriklang atteignent 90 pour cent de F1 macro là où le meilleur modèle testé en zéro-shot plafonne à 45 pour cent. Cette mesure porte sur cette tâche et ce corpus. Elle ne dit pas qu'un modèle est incapable de traiter le wolof.

## Section 6 — Assembler et documenter (15h05, 15 min)

> **Consigne (15 min)** — Exécuter les cellules dans l'ordre : fusion des annotations, découpage par locuteur, carte du dataset. Puis compléter les champs marqués à compléter dans la carte, c'est elle qu'on lit dans six mois.

Le nom du dossier est déjà une leçon : `taiss2026_sentiment_ee-fr_v1.0` porte l'origine, la tâche, les langues et la version. Un dossier nommé `dataset_final` ou `data2` est irretrouvable dans six mois et impossible à citer.

In [ ]:
# fusion des labels consensuels dans le manifeste nettoyé
annotated = clean.copy()
annotated["sentiment"] = annotated["id"].map(consensus) if consensus else ""
annotated["sentiment"] = annotated["sentiment"].fillna("")
annotated.to_csv(DATASET_DIR / "data" / "annotated_manifest.csv",
                 index=False, encoding="utf-8")
n_annotees = int((annotated["sentiment"] != "").sum())
print(f"{n_annotees} lignes portent un label consensuel")

import shutil
shutil.copy("data/raw_manifest.csv", DATASET_DIR / "data" / "raw_manifest.csv")
for script in ["quality_check.py", "agreement.py"]:
    shutil.copy(f"scripts/{script}", DATASET_DIR / "scripts" / script)

Le découpage se fait par locuteur, pas par ligne. Si les enregistrements de `spk_042` sont à la fois dans train et dans test, le modèle peut reconnaître la voix ou le style au lieu d'apprendre la tâche : le score de test devient un mensonge optimiste. C'est la fuite de données, et le manifeste porte `speaker_id` précisément pour la rendre évitable en trois lignes.

In [ ]:
import random

rng = random.Random(20260827)
source_splits = annotated.copy()
# une ligne sans locuteur ne peut pas garantir la règle : on l'isole dans
# train par convention, et la carte le documentera
source_splits["speaker_id"] = source_splits["speaker_id"].fillna("spk_inconnu")

locuteurs = sorted(source_splits["speaker_id"].unique())
rng.shuffle(locuteurs)
cible = {"train": 0.70, "validation": 0.15, "test": 0.15}
quotas = {k: v * len(source_splits) for k, v in cible.items()}
affectation, effectifs = {}, {"train": 0, "validation": 0, "test": 0}
for spk in locuteurs:
    n = int((source_splits["speaker_id"] == spk).sum())
    part = "spk_inconnu" == spk and "train" or min(
        effectifs, key=lambda s: (effectifs[s] + n) / quotas[s])
    affectation[spk] = part
    effectifs[part] += n

source_splits["split"] = source_splits["speaker_id"].map(affectation)
for part in ["train", "validation", "test"]:
    bloc = source_splits[source_splits["split"] == part].drop(columns=["split"])
    bloc.to_csv(DATASET_DIR / "splits" / f"{part}.csv",
                index=False, encoding="utf-8")
    print(f"{part:11} {len(bloc):4} lignes, {bloc.speaker_id.nunique()} locuteurs")

recouvrement = (set(source_splits[source_splits.split == "train"].speaker_id)
                & set(source_splits[source_splits.split == "test"].speaker_id))
print("locuteurs présents dans train et test :", recouvrement or "aucun")

In [ ]:
# la carte du dataset : remplie avec ce qui se mesure, à compléter pour ce
# qui se décide (licence, limites que vous connaissez et pas nous)
from datetime import date

LICENCE = "[À COMPLÉTER : ex. CC-BY-4.0]"
AUTEURS = "[À COMPLÉTER : votre binôme]"

gabarit = (Path("templates") / "dataset_card_template.md").read_text(encoding="utf-8")
carte = (gabarit
         .replace("{{origine}}", "Atelier TAISS 2026, Lomé — corpus vocal Afriklang")
         .replace("{{langues}}", "français (fr), éwé (ee)")
         .replace("{{n_items}}", str(len(annotated)))
         .replace("{{n_annotees}}", str(n_annotees))
         .replace("{{tache}}", "analyse de sentiment, trois classes")
         .replace("{{guide}}", "annotation_guide_v2.md (v1 conservée pour audit)")
         .replace("{{n_annotateurs}}", "2 par binôme, consensus simple")
         .replace("{{kappa_v1}}", str(kappa_v1) if kappa_v1 is not None else "non mesuré")
         .replace("{{kappa_v2}}", str(kappa_v2) if kappa_v2 is not None else "non mesuré")
         .replace("{{licence}}", LICENCE)
         .replace("{{date}}", date.today().isoformat())
         .replace("{{auteurs}}", AUTEURS))
(DATASET_DIR / "dataset_card.md").write_text(carte, encoding="utf-8")
print(carte)

In [ ]:
# le dossier que vous emportez
for f in sorted(DATASET_DIR.rglob("*")):
    if f.is_file():
        print(f.relative_to(DATASET_DIR.parent))

## Section 7 — À l'échelle, et questions (15h20, 10 min)

Ce que vous avez fait sur 60 phrases, Afriklang le fait tourner sur 24 langues avec plus de 200 contributeurs natifs. Ce qui change à l'échelle : les binômes deviennent des équipes, le consensus simple devient un arbitrage par un troisième annotateur quand les deux premiers divergent, et le guide d'annotation devient un document vivant, versionné comme du code, parce que chaque nouveau lot de données révèle un cas que personne n'avait prévu.

Ce qui ne change pas : un modèle ne peut pas être meilleur que la mesure qui l'évalue. Et cette mesure, quelqu'un l'a construite à la main.

Les participants les plus rigoureux d'aujourd'hui peuvent rejoindre le réseau de contributeurs rémunérés d'Afriklang : la feuille de contacts circule.